# VietTDR vòng 4 — đóng khoảng cách miền synth→real cho DASR

Vòng 3 đo được: SR thuần synthetic +3dB PSNR nhưng **−6~7 điểm** word acc
trên crop thật ≤16px; sửa giao thức giữ tỷ lệ chỉ lấy lại ~1.75 điểm.
Thủ phạm chính = lệch miền. Vòng này:

1. **Cặp thật tự tạo**: crop thật ≥28px làm HR (≈½ dữ liệu), tự suy giảm
   thành LR — thống kê ảnh là thật, chỉ phép suy giảm là nhân tạo.
2. Trộn 60k cặp tổng hợp (có mặt nạ dấu) + ~12k cặp thật (mặt nạ 0).
3. Hai run: M1 = mặt nạ; M2 = mặt nạ + loss thành phần (λ_rec 0.1).
4. Eval real ở CẢ HAI giao thức (giữ tỷ lệ + bóp cũ, để đối chiếu vòng 3),
   và eval lại sr_s2 cũ ở giao thức giữ tỷ lệ trên full test.

**Không train lại recognizer** — dùng checkpoint từ dataset `viettdr-ckpt`.
Tổng ~2.5h.

Add Input 4 nguồn: `viettdr-data`, code mới nhất (`code`),
`vintext-train-images`, **`viettdr-ckpt`**. GPU T4 x2.

In [ ]:
# Cell 1 — moi truong
import torch, os, multiprocessing, time
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else '*** KHONG CO GPU ***')
IS_KAGGLE = os.path.exists('/kaggle')
NPROC = max(2, multiprocessing.cpu_count())
T0 = time.time()
BUDGET_H = 9.0
def elapsed_h(): return (time.time() - T0) / 3600

In [ ]:
# Cell 2 — nap code + du lieu vao /tmp; tim checkpoint recognizer
import os, glob, shutil, time

SRC, WORK = ('/kaggle/input', '/tmp/vt') if IS_KAGGLE else \
            ('/content/drive/MyDrive', '/content/run')
OUTDIR = '/kaggle/working' if IS_KAGGLE else '.'
os.makedirs(WORK, exist_ok=True)

def find_all(root, name):
    return sorted(glob.glob(f'{root}/**/{name}', recursive=True))

cands = find_all(SRC, 'train.py')
good = []
for c in cands:
    root = os.path.dirname(c)
    if (os.path.exists(os.path.join(root, 'tools', 'make_real_pairs.py'))
            and '--aspect' in open(os.path.join(root, 'eval_sr.py'),
                                   encoding='utf-8').read()):
        good.append(c)
print('ban code:')
for c in cands:
    print(('   [DU CO] ' if c in good else '   [CU]    ') + os.path.dirname(c))
assert good, 'Thieu ban code vong 4 (make_real_pairs + --aspect)'
code_root = os.path.dirname(good[-1])
for item in ['viettdr', 'tools', 'tests', 'fonts', 'assets', 'train.py',
             'eval.py', 'train_sr.py', 'eval_sr.py', 'charset_vintext.txt']:
    s, d = os.path.join(code_root, item), os.path.join(WORK, item)
    if os.path.isdir(s):
        shutil.copytree(s, d, dirs_exist_ok=True)
    elif os.path.isfile(s):
        shutil.copy(s, d)

gt = None
for h in find_all(SRC, 'train_gt.jsonl'):
    if 'vintext' in h.lower():
        gt = h
        break
assert gt
data_dst = os.path.join(WORK, 'data', 'vintext_words')
if not os.path.exists(os.path.join(data_dst, 'train_gt.jsonl')):
    t0 = time.time()
    os.makedirs(os.path.dirname(data_dst), exist_ok=True)
    shutil.copytree(os.path.dirname(gt), data_dst)
    print(f'copy data: {time.time() - t0:.0f}s')

recs = find_all(SRC, 'rec_full_slim.pth')
assert recs, 'Thieu dataset viettdr-ckpt (rec_full_slim.pth)'
REC = recs[0]
olds = find_all(SRC, 'sr_s2_best.pth')
S2_OLD = olds[0] if olds else None
print('rec ckpt :', REC)
print('s2 cu    :', S2_OLD)
os.chdir(WORK)
!python tests/test_vietchar.py

In [ ]:
# Cell 3 — du lieu SR tron: 60k synth (co mat na) + cap that tu tao
import os, glob, json

bg_hits = glob.glob('/kaggle/input/**/im0001.jpg', recursive=True)
BG = ('--bg-dir ' + os.path.dirname(bg_hits[0])) if bg_hits else ''
corpus = 'data/vintext_words/train_gt.jsonl assets/general_dict.txt'

if not os.path.exists('data/sr_mix/pairs_gt.jsonl'):
    !python tools/gen_sr_pairs.py --out data/sr_mix --count 60000 \
        --fonts fonts --corpus {corpus} {BG} --workers {NPROC} \
        --bg-cache 40 --tone-balance
    !python tools/make_real_pairs.py --crops data/vintext_words \
        --split train --out data/sr_mix --min-h 28 --append

recs = [json.loads(l) for l in open('data/sr_mix/pairs_gt.jsonl',
                                     encoding='utf-8')]
n_real = sum(1 for r in recs if r.get('real'))
print(f'tong {len(recs)} cap | synth {len(recs) - n_real} | that {n_real} '
      f'| {elapsed_h():.2f}h')

In [ ]:
# Cell 4 — SR vong 4: M1 (mat na) va M2 (mat na + loss thanh phan 0.1)
import os, shutil

RUNS = [
    ('sr_m1', ''),
    ('sr_m2', f'--rec-ckpt {REC} --charset charset_vintext.txt --lam-rec 0.1'),
]
DONE = []
for name, flag in RUNS:
    if elapsed_h() > BUDGET_H:
        print(f'### BO QUA {name}')
        continue
    print('#' * 18, f'{name} ({elapsed_h():.2f}h)', '#' * 18, flush=True)
    !python train_sr.py --data data/sr_mix --out runs/{name} \
        --epochs 12 --bs 256 --workers {NPROC} {flag}
    if os.path.exists(f'runs/{name}/best.pth'):
        DONE.append(name)
        shutil.copy(f'runs/{name}/best.pth', f'{OUTDIR}/{name}_best.pth')
        shutil.copy(f'runs/{name}/log.csv', f'{OUTDIR}/{name}_log.csv')
print('xong:', DONE)

In [ ]:
# Cell 5 — danh gia: synth + real (CA HAI giao thuc) + sr_s2 cu de doi chieu
import os

for name in DONE:
    ck = f'runs/{name}/best.pth'
    print('=' * 14, name, 'synth', '=' * 14, flush=True)
    !python eval_sr.py synth --data data/sr_mix --ckpt {ck} \
        --rec-ckpt {REC} --charset charset_vintext.txt --limit 3000
    print('=' * 14, name, 'real <=16px GIU TY LE', '=' * 14, flush=True)
    !python eval_sr.py real --data data/vintext_words --split test \
        --max-h 16 --ckpt {ck} --rec-ckpt {REC} \
        --charset charset_vintext.txt --aspect
    print('=' * 14, name, 'real <=16px bop 16x64 (doi chieu)', '=' * 14, flush=True)
    !python eval_sr.py real --data data/vintext_words --split test \
        --max-h 16 --ckpt {ck} --rec-ckpt {REC} \
        --charset charset_vintext.txt

if S2_OLD:
    print('=' * 14, 'sr_s2 (vong 3) real GIU TY LE full test', '=' * 14, flush=True)
    !python eval_sr.py real --data data/vintext_words --split test \
        --max-h 16 --ckpt {S2_OLD} --rec-ckpt {REC} \
        --charset charset_vintext.txt --aspect

# output gon nhe
files = [os.path.join(dp, f) for dp, _, fs in os.walk(OUTDIR) for f in fs]
print(f'\noutput {len(files)} file | tong {elapsed_h():.2f}h')
assert len(files) < 500